# Hugging Face Pipeline Demonstration
This notebook demonstrates how to use Hugging Face's `transformers` library, focusing on pipelines for various NLP tasks. We will cover the following topics:

1. Sentiment Analysis
2. Named Entity Recognition (NER)
3. Question Answering
4. Text Generation


## Installation

Ensure that you have the necessary package (`transformers`) installed. If you don't have it yet, uncomment and run the following cell:

In [1]:
!pip install transformers datasets accelerate evaluate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.2 MB/s eta 0:00:00


In [2]:
from datasets import load_dataset

In [3]:
dataset = load_dataset("HuggMachas/Sarcasm_dataset")

README.md:   0%|          | 0.00/427 [00:00<?, ?B/s]

data/train-00000-of-00001.parquet: reconstructing file:   0%|          |  0.00B / 1.10MB            

data/train-00000-of-00001.parquet: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/5250 [00:00<?, ? examples/s]

In [4]:
print(dataset["train"][0])

{'id': 866871160725794816, 'id_str': '866871160725794816', 'text': 'Triple Talaq par Burbak Kuchh nahi bolega', 'label': 'NO', 'langid': [['Triple', 'Talaq', 'par', 'Burbak', 'Kuchh', 'nahi', 'bolega'], ['en', 'hi', 'hi', 'hi', 'hi', 'hi', 'hi']]}


In [5]:
for i in range(5):
    print(dataset["train"][i]["text"])
    print(dataset["train"][i]["label"])
    print("-" * 50)

Triple Talaq par Burbak Kuchh nahi bolega
NO
--------------------------------------------------
Batao ye uss site pr se akki sir ke verdict nikaal laaye jaha he ajay ki ek bi movie hit nai
YES
--------------------------------------------------
Hindu baheno par julam bardas nahi hoga @TripleTalaq Hindu daram par lago hoga hamari Hindu baheno ki soraksa ke liye
NO
--------------------------------------------------
Naa bhai.. aisa nhi hai.. mere handle karne se bhi kuchh hona nhi hai.. politics se mera door door tak ka naata nhi hai
NO
--------------------------------------------------
#RememberingRajiv aaj agar musalman auraten triple talaq ki wajah se itni jaleel ho rahi hai to iss ka sabse bada reason Rajib gandhi hai.
NO
--------------------------------------------------


In [7]:
from transformers import AutoTokenizer

In [8]:
from transformers import AutoModelForSequenceClassification

In [9]:
model_name = "cardiffnlp/twitter-roberta-base-sentiment-latest"

In [10]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

In [42]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=2,
    ignore_mismatched_sizes=True
)

[transformers] You passed `num_labels=2` which is incompatible to the `id2label` map of length `3`.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

[transformers] RobertaForSequenceClassification LOAD REPORT from: cardiffnlp/twitter-roberta-base-sentiment-latest
Key                         | Status     |                                                                                       
----------------------------+------------+---------------------------------------------------------------------------------------
roberta.pooler.dense.weight | UNEXPECTED |                                                                                       
roberta.pooler.dense.bias   | UNEXPECTED |                                                                                       
classifier.out_proj.weight  | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3, 768]) vs model:torch.Size([2, 768])
classifier.out_proj.bias    | MISMATCH   | Reinit due to size mismatch - ckpt: torch.Size([3]) vs model:torch.Size([2])          

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect 

In [12]:
print(type(model))

<class 'transformers.models.roberta.modeling_roberta.RobertaForSequenceClassification'>


In [13]:
def convert_labels(example):
    example["label"] = 1 if example["label"] == "YES" else 0
    return example

dataset = dataset.map(convert_labels)

Map:   0%|          | 0/5250 [00:00<?, ? examples/s]

In [14]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'id_str', 'text', 'label', 'langid'],
        num_rows: 5250
    })
})


In [15]:
from collections import Counter

Counter(dataset["train"]["label"])

Counter({0: 4746, 1: 504})

In [16]:
for i in range(10):
    print("Label:", dataset["train"][i]["label"])
    print("Text :", dataset["train"][i]["text"])
    print("-" * 50)

Label: 0
Text : Triple Talaq par Burbak Kuchh nahi bolega
--------------------------------------------------
Label: 1
Text : Batao ye uss site pr se akki sir ke verdict nikaal laaye jaha he ajay ki ek bi movie hit nai
--------------------------------------------------
Label: 0
Text : Hindu baheno par julam bardas nahi hoga @TripleTalaq Hindu daram par lago hoga hamari Hindu baheno ki soraksa ke liye
--------------------------------------------------
Label: 0
Text : Naa bhai.. aisa nhi hai.. mere handle karne se bhi kuchh hona nhi hai.. politics se mera door door tak ka naata nhi hai
--------------------------------------------------
Label: 0
Text : #RememberingRajiv aaj agar musalman auraten triple talaq ki wajah se itni jaleel ho rahi hai to iss ka sabse bada reason Rajib gandhi hai.
--------------------------------------------------
Label: 0
Text : are cricket se sanyas le liya kya viru aur social service suru kardiya.khel hi bhul gaye.2 innings 0 n 0
--------------------------------

In [17]:
def tokenize(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized_dataset = dataset.map(
    tokenize,
    batched=True
)

Map:   0%|          | 0/5250 [00:00<?, ? examples/s]

In [18]:
print(tokenized_dataset["train"][0])

{'id': 866871160725794816, 'id_str': '866871160725794816', 'text': 'Triple Talaq par Burbak Kuchh nahi bolega', 'label': 0, 'langid': [['Triple', 'Talaq', 'par', 'Burbak', 'Kuchh', 'nahi', 'bolega'], ['en', 'hi', 'hi', 'hi', 'hi', 'hi', 'hi']], 'input_ids': [0, 35587, 8293, 255, 2331, 1343, 2242, 4273, 428, 677, 229, 4272, 298, 295, 15946, 5276, 4308, 102, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [20]:
decoded = tokenizer.decode(
    tokenized_dataset["train"][1]["input_ids"],
    skip_special_tokens=True
)

print(decoded)

Batao ye uss site pr se akki sir ke verdict nikaal laaye jaha he ajay ki ek bi movie hit nai


In [21]:
tokenized_dataset = tokenized_dataset.remove_columns(
    ["id", "id_str", "text", "langid"]
)

tokenized_dataset.set_format("torch")

In [23]:
import torch
import torchvision

print(torch.__version__)
print(torchvision.__version__)

2.11.0+cu128
0.26.0+cu128


In [25]:
print(tokenized_dataset["train"].format)

{'type': 'torch', 'format_kwargs': {}, 'columns': ['label', 'input_ids', 'attention_mask'], 'output_all_columns': False}


In [26]:
tokenized_dataset.reset_format()

In [27]:
print(tokenized_dataset["train"][0])

{'label': 0, 'input_ids': [0, 35587, 8293, 255, 2331, 1343, 2242, 4273, 428, 677, 229, 4272, 298, 295, 15946, 5276, 4308, 102, 2, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]}


In [28]:
split_dataset = tokenized_dataset["train"].train_test_split(
    test_size=0.2,
    seed=42
)

In [29]:
print(split_dataset)

DatasetDict({
    train: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 4200
    })
    test: Dataset({
        features: ['label', 'input_ids', 'attention_mask'],
        num_rows: 1050
    })
})


In [30]:
from transformers import TrainingArguments

In [31]:
training_args = TrainingArguments(
    output_dir="./sarcasm_model",

    # evaluate after each epoch
    eval_strategy="epoch",

    # save model after each epoch
    save_strategy="epoch",

    # learning rate
    learning_rate=2e-5,

    # batch sizes
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,

    # train 3 times over data
    num_train_epochs=3,

    # regularization
    weight_decay=0.01,

    # logging
    logging_steps=50,

    # keep best model
    load_best_model_at_end=True,

    report_to="none"
)

In [44]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split_dataset["train"],
    eval_dataset=split_dataset["test"]
)

In [45]:
trainer.train()

Epoch,Training Loss,Validation Loss
1,0.128579,0.125632
2,0.131983,0.130633
3,0.074151,0.141239


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=789, training_loss=0.1261205068861426, metrics={'train_runtime': 334.3913, 'train_samples_per_second': 37.68, 'train_steps_per_second': 2.36, 'total_flos': 828799824384000.0, 'train_loss': 0.1261205068861426, 'epoch': 3.0})

In [46]:
print(model.device)

cuda:0


In [49]:
import torch

text = "वाह! फिर से इंटरनेट बंद हो गया, क्या शानदार सेवा है"

inputs = tokenizer(
    text,
    return_tensors="pt",
    truncation=True,
    padding=True
)

# move inputs to same device as model
inputs = {
    k: v.to(model.device)
    for k, v in inputs.items()
}

with torch.no_grad():
    outputs = model(**inputs)

pred = torch.argmax(
    outputs.logits,
    dim=1
).item()

print("Prediction:", pred)

if pred == 1:
    print("Sarcastic 😏")
else:
    print("Not Sarcastic 🙂")

Prediction: 0
Not Sarcastic 🙂


In [50]:
import torch
from torch.nn.functional import softmax

with torch.no_grad():
    outputs = model(**inputs)

probs = softmax(outputs.logits, dim=1)

print(probs)

tensor([[0.9977, 0.0023]], device='cuda:0')


In [47]:
print(model.config.id2label)
print(model.config.num_labels)

{0: 'LABEL_0', 1: 'LABEL_1'}
2


In [40]:
print(trainer.model.config.num_labels)
print(trainer.model.config.id2label)

3
{0: 'negative', 1: 'neutral', 2: 'positive'}


In [41]:
print(model.classifier.out_proj.weight.shape)

torch.Size([3, 768])


The `pipeline` method in Hugging Face allows easy access to pre-trained models for tasks like sentiment analysis, text generation, and more.

In [43]:
print(model.config.num_labels)
print(model.classifier.out_proj.weight.shape)

2
torch.Size([2, 768])


In [51]:
test_cases = [
    "वाह! फिर से बिजली चली गई, क्या शानदार व्यवस्था है",
    "Great! Demo se 2 minute pehle laptop crash ho gaya",
    "Bahut badhiya, 4 ghante traffic me phas gaya",
    "Aaj mausam bahut accha hai",
    "Mujhe cricket pasand hai",
    "Wah bhai, kya service hai, pura din network hi nahi aaya 😂",
    "Bahut khushi hui jaan kar ki salary fir delay ho gayi"
]

for text in test_cases:

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)
    pred = torch.argmax(probs, dim=1).item()

    print("\nText:", text)
    print("Prediction:", pred)
    print("Probabilities:", probs.cpu().numpy())
    print("-" * 60)


Text: वाह! फिर से बिजली चली गई, क्या शानदार व्यवस्था है
Prediction: 0
Probabilities: [[0.99760973 0.00239027]]
------------------------------------------------------------

Text: Great! Demo se 2 minute pehle laptop crash ho gaya
Prediction: 0
Probabilities: [[0.9971692  0.00283087]]
------------------------------------------------------------

Text: Bahut badhiya, 4 ghante traffic me phas gaya
Prediction: 0
Probabilities: [[0.9984659  0.00153416]]
------------------------------------------------------------

Text: Aaj mausam bahut accha hai
Prediction: 0
Probabilities: [[0.99838006 0.00161997]]
------------------------------------------------------------

Text: Mujhe cricket pasand hai
Prediction: 0
Probabilities: [[0.9985532  0.00144681]]
------------------------------------------------------------

Text: Wah bhai, kya service hai, pura din network hi nahi aaya 😂
Prediction: 0
Probabilities: [[0.9974855 0.0025145]]
------------------------------------------------------------

Text: 

In [64]:
test_cases = [

    # Expected: Sarcastic 😏
    "Wah kya baat hai, 6 ghante train late hai #Irony",
    "Maiden shabash bas ab aise hi khelo #Sarcasm",
    "Bahut badhiya banking service, 3 din se server down hai #Irony",
    "Aaj dhoni ko gaali dene wale kal usko god bolenge #Irony",
    "Bahut badhiya, salary fir delay ho gayi",
    "Wah bhai, poora din network nahi aaya, kya service hai",
    "Great! Laptop demo se pehle crash ho gaya",
    "Kamaal hai, meeting se 5 minute pehle internet band ho gaya",

    # Expected: Not Sarcastic 🙂
    "Aaj mausam bahut accha hai",
    "Mujhe cricket pasand hai",
    "Main kal Delhi ja raha hoon",
    "Bharat ne match jeet liya",
    "Aaj office mein bahut kaam tha",
    "Bachche park mein khel rahe hain",
    "Mujhe chai peena pasand hai",
    "Aaj bahut garmi hai"
]

In [65]:
for text in test_cases:

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    inputs = {
        k: v.to(model.device)
        for k, v in inputs.items()
    }

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)

    pred = torch.argmax(probs, dim=1).item()

    print("\nText:", text)
    print("Probabilities:", probs.cpu().numpy())
    print("Prediction:", pred)

    if pred == 1:
        print("Sarcastic 😏")
    else:
        print("Not Sarcastic 🙂")

    print("-" * 60)


Text: Wah kya baat hai, 6 ghante train late hai #Irony
Probabilities: [[0.22276829 0.77723175]]
Prediction: 1
Sarcastic 😏
------------------------------------------------------------

Text: Maiden shabash bas ab aise hi khelo #Sarcasm
Probabilities: [[0.26484713 0.73515284]]
Prediction: 1
Sarcastic 😏
------------------------------------------------------------

Text: Bahut badhiya banking service, 3 din se server down hai #Irony
Probabilities: [[0.21985272 0.78014725]]
Prediction: 1
Sarcastic 😏
------------------------------------------------------------

Text: Aaj dhoni ko gaali dene wale kal usko god bolenge #Irony
Probabilities: [[0.21229813 0.7877019 ]]
Prediction: 1
Sarcastic 😏
------------------------------------------------------------

Text: Bahut badhiya, salary fir delay ho gayi
Probabilities: [[0.9985781  0.00142196]]
Prediction: 0
Not Sarcastic 🙂
------------------------------------------------------------

Text: Wah bhai, poora din network nahi aaya, kya service hai
Proba

In [58]:
preds = trainer.predict(split_dataset["test"])

In [59]:
import numpy as np

predictions = np.argmax(
    preds.predictions,
    axis=1
)

print("Predicted labels:")
print(np.bincount(predictions))

Predicted labels:
[910 140]


In [61]:
count = 0

for row in dataset["train"]:
    if row["label"] == 1:
        print(row["text"])
        print("-" * 60)

        count += 1

        if count == 10:
            break

Batao ye uss site pr se akki sir ke verdict nikaal laaye jaha he ajay ki ek bi movie hit nai
------------------------------------------------------------
Abusing tweet ke liye Account suspend karte ho...   Woh bhi wo desh me... Jahan gali gali me yahi language use hota hai...   #Irony
------------------------------------------------------------
kaafiron' ko 'masjid' mein 'namaz' perhtay huaay maara gyaa..bohat khoob,mashallah! #irony
------------------------------------------------------------
@jalanjalans  Bhai AAP apne sab Kharche ka Hisaab Deti hai.. Sab Jagah.. u r saying jo Party ka 80% ka pata nahi wo hisaab deti hI
------------------------------------------------------------
Aaj dhoni ko god, magic man bolnewale kabhi usiko gaali de rahe the.. Roflol!  #Irony
------------------------------------------------------------
Saare baghwaan apne ghar pe hai. Chaar dham ki yatra mat karo. Uttrakhand mat jao. - Aunty who was in Uttrakhand to meet God. #Irony
-----------------------------

In [66]:
sarcastic_rows = [r for r in dataset["train"] if r["label"] == 1]

for row in sarcastic_rows[:10]:
    print(row["text"])


Batao ye uss site pr se akki sir ke verdict nikaal laaye jaha he ajay ki ek bi movie hit nai
Abusing tweet ke liye Account suspend karte ho...   Woh bhi wo desh me... Jahan gali gali me yahi language use hota hai...   #Irony
kaafiron' ko 'masjid' mein 'namaz' perhtay huaay maara gyaa..bohat khoob,mashallah! #irony
@jalanjalans  Bhai AAP apne sab Kharche ka Hisaab Deti hai.. Sab Jagah.. u r saying jo Party ka 80% ka pata nahi wo hisaab deti hI
Aaj dhoni ko god, magic man bolnewale kabhi usiko gaali de rahe the.. Roflol!  #Irony
Saare baghwaan apne ghar pe hai. Chaar dham ki yatra mat karo. Uttrakhand mat jao. - Aunty who was in Uttrakhand to meet God. #Irony
Maiden shabash bas ab aisay he khelo -.- #Sarcasm
#Reliance Arey Aap ki company ki advertisement kum ho gai ?Aisa mat kariyo,aap thode aur lutt lo aam janta ko lekin top par raho #irony #pun
Main bachata raha deemak se ghar apna..aur kuch kursi ke keedey poora desh khaa gaye.. :/ #Pappu #irony #maunibaba #India
Kam se kam Jai shree 

In [67]:
text = sarcastic_rows[0]["text"]
print(text)

Batao ye uss site pr se akki sir ke verdict nikaal laaye jaha he ajay ki ek bi movie hit nai


In [69]:
for row in sarcastic_rows[:10]:

    text = row["text"]

    inputs = tokenizer(
        text,
        return_tensors="pt",
        truncation=True,
        padding=True
    )

    inputs = {k: v.to(model.device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = model(**inputs)

    probs = torch.softmax(outputs.logits, dim=1)

    pred = torch.argmax(probs, dim=1).item()

    print("\nText:", text)
    print("Actual Label:", row["label"])
    print("Predicted:", pred)
    print("Probabilities:", probs.cpu().numpy())
    print("-" * 70)


Text: Batao ye uss site pr se akki sir ke verdict nikaal laaye jaha he ajay ki ek bi movie hit nai
Actual Label: 1
Predicted: 0
Probabilities: [[0.9986243  0.00137567]]
----------------------------------------------------------------------

Text: Abusing tweet ke liye Account suspend karte ho...   Woh bhi wo desh me... Jahan gali gali me yahi language use hota hai...   #Irony
Actual Label: 1
Predicted: 1
Probabilities: [[0.19240396 0.807596  ]]
----------------------------------------------------------------------

Text: kaafiron' ko 'masjid' mein 'namaz' perhtay huaay maara gyaa..bohat khoob,mashallah! #irony
Actual Label: 1
Predicted: 1
Probabilities: [[0.20280682 0.79719317]]
----------------------------------------------------------------------

Text: @jalanjalans  Bhai AAP apne sab Kharche ka Hisaab Deti hai.. Sab Jagah.. u r saying jo Party ka 80% ka pata nahi wo hisaab deti hI
Actual Label: 1
Predicted: 0
Probabilities: [[0.9984012 0.0015988]]
----------------------------------

---

## 1. Sentiment Analysis with Hugging Face Pipelines
In this section, we'll use the sentiment analysis pipeline, which analyzes whether a given text expresses a positive or negative sentiment.

In [ ]:
# Create a sentiment-analysis pipeline
classifier = pipeline('sentiment-analysis', model='distilbert-base-uncased-finetuned-sst-2-english')

# Analyze sentiment of the given text
result = classifier("I love using Hugging Face transformers!")

config.json:   0%|          | 0.00/629 [00:00<?, ?B/s]

c:\Users\rkura\anaconda3\envs\Huggingface_env\Lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rkura\.cache\huggingface\hub\models--distilbert-base-uncased-finetuned-sst-2-english. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

c:\Users\rkura\anaconda3\envs\Huggingface_env\Lib\site-packages\transformers\tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [ ]:
print(f"Sentiment Analysis: {result}")

Sentiment Analysis: [{'label': 'POSITIVE', 'score': 0.9971315860748291}]


The model confidently predicts a positive sentiment with a score of 0.997.

---

## 2. Named Entity Recognition (NER)

We can use the `ner` pipeline to identify named entities (e.g., organizations, locations) in a text.

In [ ]:
# Create a NER pipeline
ner = pipeline("ner", model='dbmdz/bert-large-cased-finetuned-conll03-english', aggregation_strategy="simple")

# Identify named entities in the text
result = ner("Apple is looking at buying a startup in San Francisco.")

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
print(f"NER: {result}")

NER: [{'entity_group': 'ORG', 'score': 0.9992366, 'word': 'Apple', 'start': 0, 'end': 5}, {'entity_group': 'LOC', 'score': 0.99951744, 'word': 'San Francisco', 'start': 40, 'end': 53}]


The model correctly identifies "Apple" as an organization (ORG) and "San Francisco" as a location (LOC), both with high confidence.

---

## 3. Question Answering

We can use the `question-answering` pipeline to extract answers from a given context based on a question.

In [ ]:
# Create a question-answering pipeline
qa_pipeline = pipeline("question-answering", model='distilbert-base-cased-distilled-squad')

# Provide a question and context
result = qa_pipeline({
    'question': "Where is Hugging Face based?",
    'context': "Hugging Face is based in New York City."
})

config.json:   0%|          | 0.00/473 [00:00<?, ?B/s]

c:\Users\rkura\anaconda3\envs\Huggingface_env\Lib\site-packages\huggingface_hub\file_download.py:159: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\rkura\.cache\huggingface\hub\models--distilbert-base-cased-distilled-squad. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to see activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/213k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/436k [00:00<?, ?B/s]

In [ ]:
print(f"Question Answering: {result}")

Question Answering: {'score': 0.9694607853889465, 'start': 25, 'end': 38, 'answer': 'New York City'}


The model accurately identifies "New York City" as the answer with a confidence score of 0.969.

---

## 4. Text Generation

The `text-generation` pipeline can generate text based on a given prompt. By default, it uses the GPT-2 model if no other model is specified.

In [ ]:
# Create a text generation pipeline
generator = pipeline("text-generation", model='gpt2')

# Generate two sequences of text, each with a maximum length of 40 tokens
results = generator("Once upon a time,", max_length=40, num_return_sequences=2)

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.


In [ ]:
for result in results:
    print(f"Text Generation: {result}")

Text Generation: {'generated_text': 'Once upon a time, we do not know how to communicate properly to someone that a system that has such information would send. And these are times of transition. Whether and how they learn of a message'}
Text Generation: {'generated_text': 'Once upon a time, when God would call the universe to a halt by throwing a bomb or by shooting somebody, we would be confronted with what comes to pass when our species, in a state of'}


In [ ]:
# Display the number of parameters in the model
print(f"Model parameter count: {generator.model.num_parameters():,}")

Model parameter count: 124,439,808


This number represents the trainable parameters in the base version of the GPT-2 model. Larger versions of GPT-2, such as GPT-2 Medium, Large, and XL, have significantly more parameters, allowing them to capture more complex patterns and generate higher-quality text but requiring more computational resources.